# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I chose Lane 4 because I want to build a working, real-world application—a system that helps an editor identify pages with low click-through rates.

The starter dataset already includes the columns needed for this task (impressions_90d, clicks_90d, ctr, avg_position, position_tier), so I can get started right away. 

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision: Which page the editor should review first.

Action: The editor rewrites the title or meta description of the page they see on the list, or edits the content so that it better matches what the user is searching for.

Cost — false positive: If a page that’s actually performing well ends up on the list, the editor wastes time and may end up breaking something that works well.

Cost — false negative: If a page that’s truly underperforming doesn’t make it onto the list at all, that opportunity is missed without being noticed.

Why ML, why not a hard rule: A single-line rule (“good position but low CTR”) isn’t sufficient because “low” means a different number for each position—a page in the 1st position might have a normal CTR of 30%, while one in the 9th position might have 3%. A fixed threshold leads to incorrect comparisons.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Depending on the position tier, the average CTR ranges from 5.5% (deep) to 35.5% (page_1)—a sixfold difference—which demonstrates why a tier-normalized approach is necessary rather than a fixed CTR threshold. 

Of the 22,006 eligible pages, 14,797 fall below their tier’s average, with the most extreme examples being pages on page_1 that receive no clicks at all—these are precisely the types of pages the editor should examine first.m

In [4]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(df.shape)

eligible = df[df['impressions_90d'] >= 100]
print(len(eligible), "pages have enough impressions to be eligible for review.")

expected_ctr = eligible.groupby('position_tier')['ctr'].mean()
print(expected_ctr)

eligible = eligible.copy()
eligible['expected_ctr'] = eligible['position_tier'].map(expected_ctr)
eligible['ctr_gap'] = eligible['expected_ctr'] - eligible['ctr']

print("Total number of pages eligible for analysis:", len(eligible))
print("Number of pages with a positive average CTR gap (i.e., below expectations):", (eligible['ctr_gap'] > 0).sum())
print("The 5 worst tier-normalized gaps:")
print(eligible.sort_values('ctr_gap', ascending=False)[['content_id','position_tier','ctr','expected_ctr','ctr_gap']].head())

(30000, 44)
22006 pages have enough impressions to be eligible for review.
position_tier
deep        0.055415
page_1      0.354760
page_3_5    0.142359
striking    0.255782
top_3       0.334128
Name: ctr, dtype: float64
Total number of pages eligible for analysis: 22006
Number of pages with a positive average CTR gap (i.e., below expectations): 14797
The 5 worst tier-normalized gaps:
                 content_id position_tier  ctr  expected_ctr  ctr_gap
8939   content_478f26883850        page_1  0.0       0.35476  0.35476
29977  content_c87291853cab        page_1  0.0       0.35476  0.35476
29983  content_6880eb215048        page_1  0.0       0.35476  0.35476
8873   content_268b56dc0221        page_1  0.0       0.35476  0.35476
8915   content_e0ae71489787        page_1  0.0       0.35476  0.35476


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim: In this sample, this many pages have a CTR below the average for their position group; 
this makes them candidates worth examining.

What I cannot claim: That the exact cause of the low CTR is the title/meta—this is merely a signal;
an editor needs to review it. Furthermore, this data does not tell me “why” it is low, only “how much” lower it is.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.